# Ai Research Assistant

#### Tools Available for AI :
* Calculator
* Wikipedia
* arxiv
* python code executor (maybe later) 
* Chat History

##### Model used : `llama-3.1-8b-instant` Groq basically


In [31]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import wikipedia
import arxiv
from langchain_core.messages import HumanMessage,AIMessage

In [32]:
load_dotenv()

True

In [33]:
chat_model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3,
    max_tokens=512,
)

In [34]:
def calc_tool (expression):
    try : 
        return (str(eval(expression)))
    except Exception:
        return "Invalid Expression"

In [35]:
def wiki_tool(text):
    try :
        return wikipedia.summary(text,
                                 sentences = 3)
    except Exception :
        return "No source Found"

In [36]:


def arxiv_tool(query):
    try:
        client = arxiv.Client()

        search = arxiv.Search(
            query=query,
            max_results=3,
            sort_by=arxiv.SortCriterion.Relevance
        )

        papers = []

        for result in client.results(search):

            papers.append(
                f"""
Title: {result.title}

Authors: {", ".join(author.name for author in result.authors)}

Published: {result.published.date()}

Summary:
{result.summary[:300]}...

Link:
{result.entry_id}
"""
            )

        return "\n\n".join(papers)

    except Exception as e:
        return f"Error: {e}"

# print(search_arxiv("Retrieval Augmented Generation"))

In [37]:
chat_history = []

In [38]:
print("AI Research Assistant Started")
print("exit to quit")
while True:
    user_input = input("You : ")
    if user_input.lower()=="exit":
        break


    tool_prompt = f"""
You are a tool selector.

Previous Conversation:
{chat_history}

Available Tools:

1. calculator
Use for:
- math
- arithmetic
- calculations

2. wikipedia
Use for:
- people
- places
- history
- concepts
- factual information

3. arxiv
Use for:
- research papers
- machine learning papers
- AI papers
- scientific papers
- requests like:
  "find papers"
  "research papers on"
  "latest papers"

4. none
Use when no tool is needed.

User Question:
{user_input}

Reply with ONLY ONE WORD:

calculator
wikipedia
arxiv
none
"""
    tool_responce = chat_model.invoke(
        [HumanMessage(content = tool_prompt)]
    )
    selected_tool = tool_responce.content.strip().lower()
    print(f"\nSelected Tool :{selected_tool} ")

    tool_result = ""
    if "calculator" in selected_tool:
        tool_result = calc_tool(user_input)

    elif "wikipedia" in selected_tool:
        tool_result = wiki_tool(user_input)

    elif "arxiv" in selected_tool:
        tool_result = arxiv_tool(user_input)

    final_prompt =  f"""
You are an AI Research Assistant.

Previous Conversation:
{chat_history}

User Question:
{user_input}

Tool Output:
{tool_result}

Instructions:

1. Use tool output when available.
2. If no tool output exists, say "####Aachaa tool nahi hai####"
3. Be concise.
4. If research papers are found, summarize them.
5. Explain technical topics clearly.
"""
    response = chat_model.invoke(
        [HumanMessage(content=final_prompt)]
    )
    print("\nAssistant:")
    print(response.content)
    print()

    chat_history.append(
        HumanMessage(content=user_input)
    )
    chat_history.append(response.content)

AI Research Assistant Started
exit to quit

Selected Tool :none 

Assistant:
####Aachaa tool nahi hai####

Since there is no previous chat history, I don't have any information to provide. Our conversation will start from scratch. What would you like to talk about?


Selected Tool :wikipedia 

Assistant:
####Aachaa tool nahi hai####

However, based on general knowledge, I can tell you that a laptop is a portable personal computer (PC). It typically has a clamshell form factor with a flat-panel screen on the inside of the upper lid and an alphanumeric keyboard and pointing device on the inside of the lower lid.


Selected Tool :wikipedia 

Assistant:
Mahatma Gandhi. 

#### Tool Output ####

Mohandas Karamchand Gandhi was an Indian lawyer, anti-colonial nationalist, and political ethicist who employed nonviolent resistance to lead India's independence from British rule.


Selected Tool :calculator 

Assistant:
####Aachaa tool nahi hai####

However, I can provide some general information 

In [39]:
print(chat_history)

[HumanMessage(content='give me previous chat history', additional_kwargs={}, response_metadata={}), "####Aachaa tool nahi hai####\n\nSince there is no previous chat history, I don't have any information to provide. Our conversation will start from scratch. What would you like to talk about?", HumanMessage(content='what is laptop', additional_kwargs={}, response_metadata={}), '####Aachaa tool nahi hai####\n\nHowever, based on general knowledge, I can tell you that a laptop is a portable personal computer (PC). It typically has a clamshell form factor with a flat-panel screen on the inside of the upper lid and an alphanumeric keyboard and pointing device on the inside of the lower lid.', HumanMessage(content='mahatma ghandhi', additional_kwargs={}, response_metadata={}), "Mahatma Gandhi. \n\n#### Tool Output ####\n\nMohandas Karamchand Gandhi was an Indian lawyer, anti-colonial nationalist, and political ethicist who employed nonviolent resistance to lead India's independence from Britis